In [1]:
import re
import math
import time
import spacy
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm
tqdm.pandas()
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from collections import Counter

C:\Users\batti\AppData\Roaming\Python\Python39\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the spaCy model
nlp = spacy.load("en_core_web_sm"
                 , 
                 disable=[
                     "ner", 
                     "parser"
                 ]
                )  # Disable unnecessary components

In [3]:
# Defining the variables

# path to the dataset
table_path = 'moviereviews.tsv'

# defining all the scenarios as list of dict.
scenarios = [
    # Please comment the scenarios which you don't want to execute.
    
    # Without lemmatization, removal of stop words, or handling logical negation
    {'lemmatize_words': False, 'remove_stop_words': False, 'handle_logical_negation': False}, 
    
    # With lemmatization
    {'lemmatize_words': True, 'remove_stop_words': False, 'handle_logical_negation': False},
    
    # With lemmatization and removal of stop words
    {'lemmatize_words': True, 'remove_stop_words': True, 'handle_logical_negation': False},
    
    # With lemmatization, removal of stop words, and handling logical negation
    {'lemmatize_words': True, 'remove_stop_words': True, 'handle_logical_negation': True}
]

df_scenario_dict = {}

# Dictionary to store train/test splits for all scenarios
df_scenario_split_dict = {}

In [4]:
# Reading the dataset from the directory.
def read_data(table_path):
    
    # Reading moviereviews table
    df = pd.read_table(table_path)
    
    return df

def data_cleanup(df):
    
    # Step 1: remove null values
    df.dropna(subset=['review'], inplace=True)
    # print(f'Number of reviews after removing null values: {df.shape[0]}')
    
    # Step 2: remove duplicates
    df.drop_duplicates(inplace=True)
    # print(f'Number of reviews after removing duplicates: {df.shape[0]}')
    
    # Step 3: Convert labels to numerical values, positive - 1, negative - 0
    df['label'] = df['label'].map({'pos': 1, 'neg': 0})
    
    return df

def preprocess_text(text, lemmatize_words=True, remove_stop_words=True, handle_logical_negation=True):
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()  # Replace multiple spaces with a single space and strip leading/trailing spaces
    
    # Process the text with SpaCy
    doc = nlp(text)
    
    if lemmatize_words:
        # Lemmatize the text
        text = ' '.join(token.lemma_ for token in doc)
    else:
        text = ' '.join(token.text for token in doc)

    if remove_stop_words:
        # Remove stop words
        text = ' '.join(token.lemma_ for token in doc if not token.is_stop)
    
    if handle_logical_negation and lemmatize_words:
        tokens = text.split()
        processed_tokens = []
        negation = False
        
        for token in tokens:
            # Check if the token is a logical negation word
            if token.lower() in ["not", "no", "never", "n't"]:
                negation = True
                processed_tokens.append(token)  # Add the negation token itself
            # If the token is punctuation, stop negation
            elif re.match(r'[.!?,]', token):
                negation = False
                processed_tokens.append(token)
            # If negation is active, prepend "NOT" to the word
            elif negation:
                processed_tokens.append(f"NOT_{token}")
            else:
                processed_tokens.append(token)
        
        text = ' '.join(processed_tokens)
    
    # Remove symbols (punctuation and other non-word characters)
    text = re.sub(r'[^\w\s]', '', text)  # Keep only words and spaces
    
    return text


In [5]:
# Reading moviereviews table to a dataframe df
df = read_data(table_path)

In [6]:
df.head()

,label,review
0,neg,how do films like mouse hunt get into theatres...
1,neg,some talented actresses are blessed with a dem...
2,pos,this has been an extraordinary year for austra...
3,pos,according to hollywood movies made in last few...
4,neg,my first press screening of 1998 and already i...


In [7]:
df['review'][0]

'how do films like mouse hunt get into theatres ? \nisn\'t there a law or something ? \nthis diabolical load of claptrap from steven speilberg\'s dreamworks studio is hollywood family fare at its deadly worst . \nmouse hunt takes the bare threads of a plot and tries to prop it up with overacting and flat-out stupid slapstick that makes comedies like jingle all the way look decent by comparison . \nwriter adam rifkin and director gore verbinski are the names chiefly responsible for this swill . \nthe plot , for what its worth , concerns two brothers ( nathan lane and an appalling lee evens ) who inherit a poorly run string factory and a seemingly worthless house from their eccentric father . \ndeciding to check out the long-abandoned house , they soon learn that it\'s worth a fortune and set about selling it in auction to the highest bidder . \nbut battling them at every turn is a very smart mouse , happy with his run-down little abode and wanting it to stay that way . \nthe story alter

In [8]:
df.shape

# Observations
# There are total 2000 reviews. 
# All reviews are categorized into 2 types.('neg', 'pos')

(2000, 2)

In [9]:
# Check for null values in each column
null_values = df.isnull().sum()
print(f"{'Column Name':<15}{'Number of Null Values':<20}")
print(f"{'-'*35}")

for column, null_count in null_values.items():
    print(f"{column:<15}{null_count:<20}")


# Observations
# Out of 2000 reviews, 35 are null.
# Since we don't have any info in these we can drop null reviews.

Column Name    Number of Null Values
-----------------------------------
label          0                   
review         35                  


In [10]:
# doing data cleaning for the whole data as it is common step for the whole dataset.
df_cleaned = data_cleanup(df)

In [11]:
df_cleaned.shape

(1940, 2)

In [12]:
# Apply preprocess_text with tqdm progress bar
for idx, params in enumerate(scenarios, start=1):
    df_scenario = df_cleaned.copy()
    df_scenario['review'] = df_scenario['review'].progress_apply(lambda x: preprocess_text(
        x, 
        lemmatize_words=params['lemmatize_words'], 
        remove_stop_words=params['remove_stop_words'], 
        handle_logical_negation=params['handle_logical_negation']
    ))
    
    df_scenario_dict[f'df_scenario{idx}'] = df_scenario

100%|██████████████████████████████████████████████████████████████████████████████| 1940/1940 [01:12<00:00, 26.67it/s]


In [13]:
df_scenario_dict['df_scenario1'].head()

,label,review
0,0,how do films like mouse hunt get into theatres...
1,0,some talented actresses are blessed with a dem...
2,1,this has been an extraordinary year for austra...
3,1,according to hollywood movies made in last few...
4,0,my first press screening of 1998 and already i...


In [14]:
df_scenario_dict['df_scenario1'].shape

(1940, 2)

In [15]:
# Splits a DataFrame into train and test sets.
def split_train_test(df, test_size=0.2, random_state=42):
    
    train, test = train_test_split(df, test_size=test_size, random_state=random_state)
    return train, test

# After splitting, limit test set to 10 rows in the next step
def slice_test_set(test_df, sample_size):
    return test_df.head(sample_size)

# Loop through scenario results and apply the split function
for scenario_name, scenario_df in df_scenario_dict.items():
    print(scenario_name)
    train_df, test_df = split_train_test(scenario_df)
    
    # Slice the test set to only 10 rows after the split
    # test_df = slice_test_set(test_df, 20)
    
    df_scenario_split_dict[scenario_name] = {
        'train': train_df,
        'test': test_df
    }
    

df_scenario1
df_scenario2
df_scenario3
df_scenario4


In [16]:
# This function will return all the unique words and count of unique words in the training dataset.
def vocabulary_size(df):   
    # Combine all the reviews into a single string
    combined_reviews = ' '.join(df['review'])

    # Split the string into individual words
    words = combined_reviews.split()

    # Convert the list of words into a set to keep only unique words
    unique_words = list(set(words))
    
    # count the total number of words present in the vocabulary
    total_words = len(unique_words)
    
    return unique_words, total_words

# Calculate prior class probabilities - p(pos) & p(neg)
def calculate_prior_probabilities(df, label_column):
    # Get total number of reviews
    total_reviews = len(df)

    # Get unique classes and their counts
    class_counts = df[label_column].value_counts()

    # Calculate prior probabilities for each class
    prior_probabilities = class_counts / total_reviews
    
    # Convert to a dictionary with keys 'prob_neg' and 'prob_pos'
    prob_dict = {
        'prob_neg': prior_probabilities.get(0, 0),  # 0 for negative, default to 0 if not found
        'prob_pos': prior_probabilities.get(1, 0)   # 1 for positive, default to 0 if not found
    }
    
    return prob_dict


# Calculate the data set size
def each_class_vocabulary_size(df, label):   
    # Combine all the reviews into a single string
    combined_reviews = ' '.join(df[df['label'] == label]['review'])

    # Split the string into individual words
    words = list(combined_reviews.split())
    
    # count the total number of words present in the vocabulary
    total_words = len(words)
    
    return words, total_words

# Train Naive Bayes Classifier - This function helps us in calculating all the required metrics.
def train_nb_classifier(df):
    vocab, V = vocabulary_size(df)
    # print(V)
    prior_probabilities = calculate_prior_probabilities(df, 'label')
    # print(prior_probabilities)
    
    pos_vocab, pos_vocab_size = each_class_vocabulary_size(df, 1)
    neg_vocab, neg_vocab_size = each_class_vocabulary_size(df, 0)
    
    # Create and return the dictionary with all the results
    results = {
        'vocab': vocab,
        'vocab_size': V,
        'prior_probabilities': prior_probabilities,
        'pos_vocab': pos_vocab,
        'pos_vocab_size': pos_vocab_size,
        'neg_vocab': neg_vocab,
        'neg_vocab_size': neg_vocab_size
    }
    # print(results)
    return results

In [17]:
# Initialize a dictionary to store the Naive Bayes training results for each scenario
nb_training_results = {}

# Loop through each scenario in df_scenario_split_dict
for scenario_name, scenario_data in df_scenario_split_dict.items():
    # Extract the training dataset
    train_df = scenario_data['train']
    
    # Apply the train_nb_classifier function to the training dataset
    results = train_nb_classifier(train_df)
    
    # Store the results in nb_training_results using the scenario name as the key
    nb_training_results[scenario_name] = results


In [18]:
nb_training_results['df_scenario1']['vocab_size']

35271

In [19]:
# Filter the test set to remove unknown words
def filter_unknown_words(text, vocab):
    return ' '.join(word for word in text.split() if word in vocab)

# Count occurrences of words in vocabularies
def count_word_occurrences(text, positive_counts, negative_counts):
    words = text.split()
    
    # Create counts for the words in the review
    pos_count = {word: 0 for word in words}
    neg_count = {word: 0 for word in words}
    
    for word in words:
        pos_count[word] = positive_counts.get(word, 0)
        neg_count[word] = negative_counts.get(word, 0)
    
    return pos_count, neg_count

# Count occurrences for each review and store in a dict
def count_word_occurrences_per_review(df, positive_vocab, negative_vocab):
    # Create counters for the positive and negative vocabularies
    positive_counts = Counter(positive_vocab)
    negative_counts = Counter(negative_vocab)
    
    result_dict = {}
    
    for index, row in tqdm(df.iterrows(), desc="Calculating word occurrences in pos and neg train vocab", total=len(df)):
        review = row['review']
        pos_counts, neg_counts = count_word_occurrences(review, positive_counts, negative_counts)
        
        # Creating a dict for each review with word counts
        result_dict[index] = {
            'review': review,
            'label': row['label'],
            'pos_counts': pos_counts,
            'neg_counts': neg_counts
        }
    
    return result_dict

# # Count occurrences of words in vocabularies
# def count_word_occurrences(text, positive_vocab, negative_vocab):
#     words = text.split()
#     pos_count = {word: 0 for word in words}
#     neg_count = {word: 0 for word in words}
    
#     for word in words:
#         pos_count[word] = positive_vocab.count(word)  
#         neg_count[word] = negative_vocab.count(word)
#         # if word in positive_vocab:
#         #     pos_count[word] += 1
#         # if word in negative_vocab:
#         #     neg_count[word] += 1
#     # print(f'positive counts: {pos_count}')
#     # print(f'negative counts: {neg_count}')
#     return pos_count, neg_count

# # Count occurrences for each review and store in a dict
# def count_word_occurrences_per_review(df, positive_vocab, negative_vocab):
#     result_dict = {}
    
#     for index, row in tqdm(df.iterrows(), desc="Calculating word occurances in pos and neg train vocab", total=len(df)):
#         review = row['review']
#         pos_counts, neg_counts = count_word_occurrences(review, positive_vocab, negative_vocab)
        
#         # Creating a dict for each review with word counts
#         result_dict[index] = {
#             'review': review,
#             'label': row['label'],
#             'pos_counts': pos_counts,
#             'neg_counts': neg_counts
#         }
    
#     return result_dict


def test_nb_classifier(df, vocab, vocab_size, prior_probabilities, pos_vocab, pos_vocab_size, neg_vocab, neg_vocab_size):
    # remove unknown words
    df['review'] = df['review'].apply(lambda x: filter_unknown_words(x, vocab))
    
    return df

In [20]:
# calculate likelihoods
def calculate_likelihood(metric_res, pos_vocab_size, neg_vocab_size, vocab_size):
    likelihood = {}
    alpha = 0.025
    
    for key, data in tqdm(metric_res.items(), desc="Calculating likelihood", total=len(metric_res)):
        review = data['review']
        label = data['label']
        pos_counts = data['pos_counts']
        neg_counts = data['neg_counts']
        
        
        # Initialize dictionaries to hold likelihood values
        pos_likelihood = {}
        neg_likelihood = {}
        
        # Calculate likelihood for each word in the review
        for word in pos_counts:
            # Calculate positive class likelihood
            pos_likelihood[word] = (pos_counts[word] + 1) / (vocab_size + pos_vocab_size)
            
        for word in neg_counts:
            # Calculate negative class likelihood
            neg_likelihood[word] = (neg_counts[word] + 1) / (vocab_size + neg_vocab_size)
        
        # Store results in the dictionary
        likelihood[key] = {
            'review': review,
            'label': label,
            'pos': pos_likelihood,
            'neg': neg_likelihood
        }
    
    return likelihood

In [21]:
def cal_sentiment_log(prior_probabilities, likelihood, df):
    sentiment = {}

    # Initialize prior probabilities for negative and positive sentiments
    prob_neg = prior_probabilities['prob_neg']
    prob_pos = prior_probabilities['prob_pos']
    
    # Iterate through each review in the likelihood dictionary
    for key, data in tqdm(likelihood.items(), desc="Calculating Log Sentiment", total=len(likelihood)):
        # Extract the likelihoods for negative and positive sentiment for this review
        neg_likelihood = data['neg']
        pos_likelihood = data['pos']

        # Initialize log probabilities with the log of the prior probabilities
        neg_sentiment = math.log(prob_neg)
        pos_sentiment = math.log(prob_pos)

        # Loop through the words and sum the log probabilities instead of multiplying
        for word in neg_likelihood:
            neg_sentiment += math.log(neg_likelihood[word])
        for word in pos_likelihood:
            pos_sentiment += math.log(pos_likelihood[word])
        
        # Extract review and label
        review = data['review']
        label = data['label']
        
        # Predict the label based on which sentiment has a higher log probability
        pred_label = 0 if neg_sentiment > pos_sentiment else 1
        # pred_label = 0 if abs(pos_sentiment) - abs(neg_sentiment) > 20 else 1
        
        # Store the result for this review
        sentiment[key] = {
            'neg_sentiment': neg_sentiment,
            'pos_sentiment': pos_sentiment,
            # 'difference': abs(pos_sentiment) - abs(neg_sentiment),
            'pred_label': pred_label
        }
        
        # Append the predicted label to the df for the corresponding review
        df.loc[key, 'neg_sentiment'] = neg_sentiment
        df.loc[key, 'pos_sentiment'] = pos_sentiment
        # df.loc[key, 'difference'] = abs(pos_sentiment) - abs(neg_sentiment)
        df.loc[key, 'pred_label'] = pred_label
        
    
    # Return the calculated sentiment and the updated DataFrame
    return sentiment, df

In [22]:
def calculate_metrics(df_test):
    # Extract true labels and predicted labels
    true_labels = df_test['label']
    pred_labels = df_test['pred_label']
    
    # Calculate confusion matrix
    conf_matrix = confusion_matrix(true_labels, pred_labels)
    
    # Calculate precision, recall, and F1 score
    precision = precision_score(true_labels, pred_labels)
    recall = recall_score(true_labels, pred_labels)
    f1 = f1_score(true_labels, pred_labels)
    
    return conf_matrix, precision, recall, f1

def print_metrics(df_class_final):
    # Calculate confusion matrix, precision, recall, and F1 score
    conf_matrix, precision, recall, f1 = calculate_metrics(df_class_final)

    # Output results
    print("Confusion Matrix:\n", conf_matrix)
    print("Precision: ", precision)
    print("Recall: ", recall)
    print("F1 Score: ", f1)

In [23]:
# Loop through each scenario in nb_training_results with a progress bar
df_results1 = {}
df_results2 = {}
metric_results1 = {} # word count given class
metric_results2 = {} # likelihood
metric_results3 = {} # sentiment

for scenario_name, results in nb_training_results.items():
    print(f"Testing scenario: {scenario_name}")
    # print(f"training results: {results}")
    # Extract the parameters from the results
    prior_probabilities = results['prior_probabilities']
    vocab = results['vocab']
    vocab_size = results['vocab_size']
    pos_vocab = results['pos_vocab']
    pos_vocab_size = results['pos_vocab_size']
    neg_vocab = results['neg_vocab']
    neg_vocab_size = results['neg_vocab_size']
    
    print(f"pos_vocab_size: {pos_vocab_size}")
    print(f"neg_vocab_size: {neg_vocab_size}")
    print(f"vocab_size: {vocab_size}")
    # Get the test DataFrame for this scenario
    df_test = df_scenario_split_dict[scenario_name]['test']
    
    # Call test_nb_classifier function
    df_test_results = test_nb_classifier(
        df_test, 
        vocab, 
        vocab_size, 
        prior_probabilities, 
        pos_vocab, 
        pos_vocab_size, 
        neg_vocab, 
        neg_vocab_size
    )
    
    # Store the test results for the current scenario
    df_results1[scenario_name] = df_test_results
    
    metric_results1[scenario_name] = count_word_occurrences_per_review(df_results1[scenario_name], pos_vocab, neg_vocab)
    
    # calculate likelihood
    metric_results2[scenario_name] = calculate_likelihood(metric_results1[scenario_name], pos_vocab_size, neg_vocab_size, vocab_size)
    
    metric_results3[scenario_name], df_results2[scenario_name] = cal_sentiment_log(prior_probabilities, metric_results2[scenario_name], df_results1[scenario_name])
    
    print_metrics(df_results2[scenario_name])

Testing scenario: df_scenario1
pos_vocab_size: 529218
neg_vocab_size: 484973
vocab_size: 35271


Calculating word occurrences in pos and neg train vocab: 100%|█████████████████████| 388/388 [00:00<00:00, 2211.85it/s]
Calculating Log Sentiment: 100%|███████████████████████████████████████████████████| 388/388 [00:00<00:00, 2820.10it/s]


Confusion Matrix:
 [[152  32]
 [ 32 172]]
Precision:  0.8431372549019608
Recall:  0.8431372549019608
F1 Score:  0.8431372549019607
Testing scenario: df_scenario2
pos_vocab_size: 529216
neg_vocab_size: 484972
vocab_size: 28108


Calculating word occurrences in pos and neg train vocab: 100%|█████████████████████| 388/388 [00:00<00:00, 2417.28it/s]
Calculating Log Sentiment: 100%|███████████████████████████████████████████████████| 388/388 [00:00<00:00, 3187.16it/s]


Confusion Matrix:
 [[153  31]
 [ 32 172]]
Precision:  0.8472906403940886
Recall:  0.8431372549019608
F1 Score:  0.8452088452088452
Testing scenario: df_scenario3
pos_vocab_size: 250633
neg_vocab_size: 226735
vocab_size: 27923


Calculating word occurrences in pos and neg train vocab: 100%|█████████████████████| 388/388 [00:00<00:00, 4240.14it/s]
Calculating Log Sentiment: 100%|███████████████████████████████████████████████████| 388/388 [00:00<00:00, 3741.49it/s]


Confusion Matrix:
 [[149  35]
 [ 33 171]]
Precision:  0.8300970873786407
Recall:  0.8382352941176471
F1 Score:  0.8341463414634146
Testing scenario: df_scenario4
pos_vocab_size: 250634
neg_vocab_size: 226735
vocab_size: 27949


Calculating word occurrences in pos and neg train vocab: 100%|█████████████████████| 388/388 [00:00<00:00, 4107.39it/s]
Calculating Log Sentiment: 100%|███████████████████████████████████████████████████| 388/388 [00:00<00:00, 3634.72it/s]

Confusion Matrix:
 [[149  35]
 [ 33 171]]
Precision:  0.8300970873786407
Recall:  0.8382352941176471
F1 Score:  0.8341463414634146


In [24]:
df_results2['df_scenario1']

,label,review,neg_sentiment,pos_sentiment,pred_label
1657,0,mulholland drive did very well at the cannes f...,-2717.331959,-2738.211613,0.0
1965,1,ok i admit i had a bad attitude about this fil...,-1869.142180,-1891.425844,0.0
70,0,it s now the anniversary of the of julie james...,-2494.511962,-2528.321510,0.0
1953,0,back in february at the monthly los angeles co...,-3183.301926,-3230.722536,0.0
1786,1,all great things come to an end and the dot co...,-2016.960525,-2008.078792,1.0
...,...,...,...,...,...
551,0,i ve never written a review for a movie i have...,-3081.564340,-3098.991164,0.0
1598,1,usually when one is debating who the modern qu...,-1476.880519,-1490.290372,0.0
546,0,you ca nt have any of this it s all mine a twi...,-4864.710584,-4876.954245,0.0
630,0,when walt disney pictures announced a live act...,-3674.728588,-3701.075165,0.0


In [25]:
# # likelihoods
# {'mulholland': 7.688699917730912e-06,
#  'drive': 8.842004905390548e-05,
#  'did': 0.0011552271626390693,
#  'very': 0.0010841066884000584,
#  'well': 0.001137927587824175,
#  'at': 0.00362906636116899,
#  'the': 0.051395114600072275,
#  'cannes': 5.766524938298183e-06,
#  'film': 0.006283590007765587,
#  'festival': 2.691044971205819e-05,
#  'as': 0.007333097546535856,
#  'you': 0.0039500695827342554,
#  'can': 0.001729957481489455,

# {'mulholland': 3,
#  'drive': 45,
#  'did': 600,
#  'very': 563,
#  'well': 591,
#  'at': 1887,
#  'the': 26737,
#  'cannes': 2,
#  'film': 3268,
#  'festival': 13,
#  'as': 3814,
#  'you': 2054,
#  'can': 899,

    
# pos_vocab_size: 529218
# neg_vocab_size: 484973
# vocab_size: 35271

# likelihood = 900/(484973+35271)
# print(likelihood)


# metric_results2['df_scenario1'][1657]['neg']

In [30]:
# metric_results1['df_scenario1'][1657]['neg_counts']

In [27]:
# metric_results2['df_scenario1']

In [28]:
# Now test_results contains the results for each scenario
# df_results['df_scenario1']['review'][1965]